# Return Codes

Return codes are the old school way to signal success or failure. Think of them as thumbs up or thumbs down after calling a function. They're not as fancy as exceptions, but the're cheap and predictable. 

````C++
#include <fstream>
#include <string> 

bool readfile(const std::string& path, std::string& out){
    std::ifstream file(path);
    if(!file){
        return false; // failure
    }
    out.assign((std::istreambuf_iterator<char>(file)),
                std::istreambuf_iterator<char());
    return true; // success
}
````

Here, instead of throwing an exception, the function just says true or false. It's simple, did it work or not? But here's the downside. All we know is that `false` equals failure. We don't know why it failed. Was the file missing? Was it locked by another program, or was there a permission problem? If you only need to know success versus failure, this pattern is fine. It's cheap, it's fast, and it avoids exceptions. But the momento you need more detail, you'll want something richer, like an error code `enum` or standard optional. Instead of returning just a `true` or `false`, we can return a numeric error code. Think of it liek ordering food at the restaurant. If the waiter just says, no, that's the Boein version. You know you didn't get your ketchup, but you don't know why. But if the waiter says, sorry, the supplier didn't deliver today or the last bottle ran out, that's the error code version. You still didn't get your ketchup, but now you know the reason. In our code, if the file can't be opened, we don't just say `false`. We return file not found. If it succeeds, then we return success. 

````C++
enum ErrorCode {SUCCESS = 0, FILE_NOT_FOUND = 1, PERMISSION_DENIED = 2};

ErrorCode read_file_code(const std::string& path, std::string& out){
    std::ifstream file(path);
    if(!file) return FILE_NOT_FOUND;
    out.assign((std::istreambuf_iterator<char>(file)),
                std::istreambuf_iterator<char>());
    return SUCCESS;
}
````

Here, you get more detail that just `true` or `false`. The caller can decide what to do based on the exact problem. But as your program grows you en up juggling lots of numeric codes. It can clutter up the code base and make rror handling repetitive. Error codes are like a more talkative waiter; they give you valuable detail. But if you use them everywhere, your code can end up full of boilerplate condition checks. Now, let's improve error codes with **enum class**. In the previous example, we used a plain `enum`. That works, but there's a problem, the names like `success`or `file not found` exist in the global scope. If another part of the program has a constant with the same name, then theey collide. That's where **enum class** comes in.  It's like putting all your error codes inside a tidy namespace of their own. Notice how the names are scoped. You have to write error code success, or error code file not found (In the code below) That make it crystal clear what you mean, and it avoids conflicts. Think of plan `enum` as having roomates who all leave their sutff in the hallway. It's easy to trip over. **Enum class** is like givin each person their own locker with a name on it. Everything stays organized, and you always know where a value belongs. **Enum class** is the modern safet version of `enums`. It gives you the clarity of error codes, but without the risk of name crashes or confusion with integers. Thisi is why in modern C++, you'll see `enum class` almost everywhere, instead of raw `enums`. 

````C++
enum class ErrorCode { Success, FileNotFound, PermissionDenied}; 

ErrorCode read_file_enum(const std::string& path, std::string& out){
    std::ifstream file(path);
    if(!file) return ErrorCode::FileNotFound; 
    out.assign((std::istreambuf_iterator<char>(file)),
                std::istreambuf_iterator<char());
    return ErrorCode::Success;
}
````